# Baseline: QLoRA Text-to-SVG Generation

**NYU Deep Learning — Spring 2026 Midterm Kaggle Competition**

This notebook implements a complete baseline pipeline in **two phases**:

**Phase A — Training** (requires internet & GPU):

1. Environment Setup — dependencies, seeds, configuration
2. SVG Utilities — competition-compliant validation, post-processing, fallback
3. Data Pipeline — multi-source loading, normalization, quality filtering
4. Model Setup — Qwen 2B + QLoRA via Unsloth
5. Training — SFT with chat-formatted (prompt, SVG) pairs

**Phase B — Inference & Submission** (can run offline on Kaggle):

6. Inference & Submission — generate, validate, export `submission.csv`

> **For Kaggle Code Submission:** split at the phase boundary. Upload the trained adapter as a Kaggle dataset, then create a separate offline notebook that only runs Phase B.

In [1]:
%pip install -q unsloth datasets trl transformers accelerate peft bitsandbytes pandas lxml cairosvg

# Mount Google Drive (Colab only)
from google.colab import drive
drive.mount('/content/drive')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.8/54.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 MB 31.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 106.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.9/402.9 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 87.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━

In [2]:
import os
import re
import time
import random
import xml.etree.ElementTree as ET
from collections import Counter

import numpy as np
import pandas as pd
import torch
from datasets import Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

ET.register_namespace('', 'http://www.w3.org/2000/svg')
ET.register_namespace('xlink', 'http://www.w3.org/1999/xlink')

print(f'Torch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [ ]:
CONFIG = {
    # Model
    'model_name': 'unsloth/Qwen2.5-3B-Instruct',
    'max_seq_length': 2048,

    # LoRA
    'lora_r': 16,
    'lora_alpha': 16,
    'lora_dropout': 0,
    'target_modules': [
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],

    # Training
    'learning_rate': 2e-4,
    'num_train_epochs': 1,
    'per_device_train_batch_size': 1,
    'gradient_accumulation_steps': 32,
    'warmup_ratio': 0.05,
    'weight_decay': 0.01,
    'logging_steps': 20,
    'eval_steps': 100,
    'save_steps': 200,

    # Data
    'max_train_samples_per_source': 12000,
    'max_svg_train_chars': 2500,
    'eval_size': 0.02,

    # SVG constraints (competition rules)
    'max_svg_length': 16000,
    'max_path_count': 256,
    'canvas_size': 256,

    # Inference
    'inference_batch_size': 4,
    'max_new_tokens': 768,
    'temperature': 0.3,
    'top_p': 0.8,
    'repetition_penalty': 1.05,

    # Paths — Colab + Google Drive
    'train_data_path': '/content/drive/MyDrive/midterm/train.csv',
    'output_dir': '/content/drive/MyDrive/midterm/qwen25_3b_svg_lora',
    'test_prompts_path': '/content/drive/MyDrive/midterm/test.csv',
    'submission_path': '/content/drive/MyDrive/midterm/submission.csv',
    # # Paths — Kaggle
    # 'train_data_path': '/kaggle/input/competitions/dl-spring-2026-svg-generation/train.csv',
    # 'output_dir': '/kaggle/working/qwen25_3b_svg_lora',
    # 'test_prompts_path': '/kaggle/input/competitions/dl-spring-2026-svg-generation/test.csv',
    # 'submission_path': '/kaggle/working/submission.csv',
}

CONFIG

{'model_name': 'unsloth/Qwen2.5-3B-Instruct',
 'max_seq_length': 2048,
 'lora_r': 16,
 'lora_alpha': 16,
 'lora_dropout': 0,
 'target_modules': ['q_proj',
  'k_proj',
  'v_proj',
  'o_proj',
  'gate_proj',
  'up_proj',
  'down_proj'],
 'learning_rate': 0.0002,
 'num_train_epochs': 1,
 'per_device_train_batch_size': 1,
 'gradient_accumulation_steps': 32,
 'warmup_ratio': 0.05,
 'weight_decay': 0.01,
 'logging_steps': 20,
 'eval_steps': 100,
 'save_steps': 200,
 'max_train_samples_per_source': 12000,
 'max_svg_train_chars': 2500,
 'eval_size': 0.02,
 'max_svg_length': 16000,
 'max_path_count': 256,
 'canvas_size': 256,
 'inference_batch_size': 4,
 'max_new_tokens': 768,
 'temperature': 0.3,
 'top_p': 0.8,
 'repetition_penalty': 1.05,
 'train_data_path': '/kaggle/input/competitions/dl-spring-2026-svg-generation/train.csv',
 'output_dir': '/kaggle/working/qwen25_3b_svg_lora',
 'test_prompts_path': '/kaggle/input/competitions/dl-spring-2026-svg-generation/test.csv',
 'submission_path': '/ka

---
## 2. SVG Utilities

Competition-compliant validation, post-processing, and fallback.

Key constraints: 256×256 canvas, ≤16 000 chars, ≤256 paths, whitelist-only tags, no scripts/events/animation.

In [4]:
ALLOWED_TAGS = frozenset({
    'svg', 'g', 'path', 'rect', 'circle', 'ellipse', 'line', 'polyline',
    'polygon', 'defs', 'use', 'symbol', 'clipPath', 'mask',
    'linearGradient', 'radialGradient', 'stop', 'text', 'tspan',
    'title', 'desc', 'style', 'pattern', 'marker', 'filter',
})


def _strip_ns(tag):
    return tag.split('}', 1)[-1] if '}' in tag else tag


def _count_paths(root):
    return sum(1 for e in root.iter() if _strip_ns(e.tag) == 'path')


def _collect_bad_tags(root):
    return {_strip_ns(e.tag) for e in root.iter() if _strip_ns(e.tag) not in ALLOWED_TAGS}


def _has_event_handlers(root):
    for e in root.iter():
        for attr in e.attrib:
            local = attr.split('}')[-1] if '}' in attr else attr
            if local.lower().startswith('on'):
                return True
    return False


def validate_svg(svg_text, max_length=16000, max_paths=256):
    """
    Competition-compliant SVG validation.
    Returns (is_valid, reason).
    """
    if not svg_text or not isinstance(svg_text, str):
        return False, 'empty'
    if len(svg_text) > max_length:
        return False, f'too_long ({len(svg_text)})'
    try:
        root = ET.fromstring(svg_text)
    except ET.ParseError as e:
        return False, f'parse_error: {e}'
    if _strip_ns(root.tag) != 'svg':
        return False, f'root_not_svg ({root.tag})'
    bad = _collect_bad_tags(root)
    if bad:
        return False, f'bad_tags: {bad}'
    if _has_event_handlers(root):
        return False, 'event_handlers'
    n = _count_paths(root)
    if n > max_paths:
        return False, f'too_many_paths ({n})'
    return True, 'ok'


def sanitize_svg(svg_text):
    """
    Post-process SVG: fix root attributes, strip disallowed elements/attributes.
    Returns cleaned SVG string or empty string on failure.
    """
    if not svg_text:
        return ''
    try:
        root = ET.fromstring(svg_text)
    except ET.ParseError:
        return ''
    if _strip_ns(root.tag) != 'svg':
        return ''
    if '{' not in root.tag:
        root.tag = '{http://www.w3.org/2000/svg}svg'
    root.attrib.pop('xmlns', None)
    root.set('width', '256')
    root.set('height', '256')
    root.set('viewBox', '0 0 256 256')
    _remove_bad_elements(root)
    _remove_event_attrs(root)
    return ET.tostring(root, encoding='unicode')


def _remove_bad_elements(elem):
    to_remove = [c for c in elem if _strip_ns(c.tag) not in ALLOWED_TAGS]
    for c in to_remove:
        elem.remove(c)
    for c in elem:
        _remove_bad_elements(c)


def _remove_event_attrs(root):
    for e in root.iter():
        bad = [a for a in e.attrib
               if (a.split('}')[-1] if '}' in a else a).lower().startswith('on')]
        for a in bad:
            del e.attrib[a]


def fallback_svg(prompt=''):
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" '
        'viewBox="0 0 256 256">'
        '<rect x="0" y="0" width="256" height="256" fill="white"/>'
        '<circle cx="128" cy="128" r="64" fill="gray"/>'
        '</svg>'
    )


_ok, _reason = validate_svg(fallback_svg())
print(f'Fallback SVG valid: {_ok} ({_reason}), length: {len(fallback_svg())} chars')

Fallback SVG valid: True (ok), length: 196 chars


---
## 3. Data Pipeline

Load competition `train.csv`, normalize SVGs to 256×256, and apply quality filters.

In [5]:
_PROMPT_PREFIX_RE = re.compile(
    r'^(?:Generate|Create|Write|Make|Draw|Produce)\s+'
    r'(?:(?:the\s+|an?\s+)?(?:SVG|svg)\s+(?:code|image|graphic)\s+)?'
    r'(?:for\s+)?(?:an?\s+image\s+(?:that|which)\s+looks?\s+like[:\s]*)?',
    re.IGNORECASE,
)
_PROMPT_SUFFIX_RE = re.compile(
    r"[\.\s]*(?:Don'?t|do\s+not)\s+use\s+markdown.*$",
    re.IGNORECASE,
)


def _clean_prompt(prompt):
    """Strip instruction wrappers, keeping only the visual description."""
    prompt = _PROMPT_PREFIX_RE.sub('', prompt)
    prompt = _PROMPT_SUFFIX_RE.sub('', prompt)
    return prompt.strip()


def _quality_filter(example):
    """Keep only samples with valid, reasonably-sized SVGs."""
    svg = example.get('svg', '')
    prompt = example.get('prompt', '')
    if not svg or not prompt:
        return False
    if len(svg) > CONFIG['max_svg_train_chars']:
        return False
    ok, _ = validate_svg(svg)
    return ok

In [6]:
# ── Load competition train.csv as primary data source ──
comp_df = pd.read_csv(CONFIG['train_data_path'])
print(f'Competition train.csv: {len(comp_df)} rows, columns={list(comp_df.columns)}')

comp_ds = Dataset.from_pandas(comp_df[['prompt', 'svg']])


def _normalize_comp(example):
    prompt = str(example.get('prompt', '')).strip()
    prompt = _clean_prompt(prompt)
    svg = sanitize_svg(str(example.get('svg', '')))
    return {'prompt': prompt, 'svg': svg}


comp_ds = comp_ds.map(_normalize_comp, remove_columns=comp_ds.column_names,
                      desc='normalizing train.csv')
before = len(comp_ds)
comp_ds = comp_ds.filter(_quality_filter, desc='filtering train.csv')
print(f'  Usable: {len(comp_ds)} rows (dropped {before - len(comp_ds)})')

max_samples = CONFIG['max_train_samples_per_source']
if max_samples and len(comp_ds) > max_samples:
    comp_ds = comp_ds.shuffle(seed=SEED).select(range(max_samples))
    print(f'  Subsampled to {len(comp_ds)} rows')

train_raw = comp_ds.shuffle(seed=SEED)
splits = train_raw.train_test_split(test_size=CONFIG['eval_size'], seed=SEED)
train_ds = splits['train']
eval_ds = splits['test']

print(f'\nTrain: {len(train_ds)} | Eval: {len(eval_ds)}')
if len(train_ds) > 0:
    print(f'Sample prompt: {train_ds[0]["prompt"][:120]}')
    print(f'Sample SVG (first 200): {train_ds[0]["svg"][:200]}')
else:
    raise RuntimeError(
        'Training set is empty after filtering. '
        'Check max_svg_train_chars / validate_svg settings.'
    )

Competition train.csv: 50000 rows, columns=['id', 'prompt', 'svg']


normalizing train.csv:   0%|          | 0/50000 [00:00<?, ? examples/s]

filtering train.csv:   0%|          | 0/50000 [00:00<?, ? examples/s]

  Usable: 319 rows (dropped 49681)

Train: 312 | Eval: 7
Sample prompt: a blue airplane icon on a black background
Sample SVG (first 200): <svg viewBox="0 0 256 256" width="256" height="256" fill="black" xmlns="http://www.w3.org/2000/svg">
  <path fill="#1f77b4" d="M12 3.5l4 4.5-4 4.5-4-4.5 4-4.5zM12 12v4.5l4-2.25-4-2.25z" />
  <path fil


In [7]:
SYSTEM_PROMPT = (
    'You generate compact, valid SVG code from text descriptions. '
    'Output only a single <svg> element with xmlns="http://www.w3.org/2000/svg", '
    'width="256", height="256", viewBox="0 0 256 256". '
    'No explanation, no markdown — only SVG code.'
)


def format_sft(example):
    text = (
        '<|im_start|>system\n'
        f'{SYSTEM_PROMPT}<|im_end|>\n'
        '<|im_start|>user\n'
        f'{example["prompt"]}<|im_end|>\n'
        '<|im_start|>assistant\n'
        f'{example["svg"]}<|im_end|>'
    )
    return {'text': text}


train_text = train_ds.map(format_sft, remove_columns=train_ds.column_names)
eval_text = eval_ds.map(format_sft, remove_columns=eval_ds.column_names)

print('Formatted sample (first 500 chars):')
print(train_text[0]['text'][:500])

Map:   0%|          | 0/312 [00:00<?, ? examples/s]

Map:   0%|          | 0/7 [00:00<?, ? examples/s]

Formatted sample (first 500 chars):
<|im_start|>system
You generate compact, valid SVG code from text descriptions. Output only a single <svg> element with xmlns="http://www.w3.org/2000/svg", width="256", height="256", viewBox="0 0 256 256". No explanation, no markdown — only SVG code.<|im_end|>
<|im_start|>user
a blue airplane icon on a black background<|im_end|>
<|im_start|>assistant
<svg viewBox="0 0 256 256" width="256" height="256" fill="black" xmlns="http://www.w3.org/2000/svg">
  <path fill="#1f77b4" d="M12 3.5l4 4.5-4 4.5-


---
## 4. Model Setup — Qwen 2B + QLoRA (Unsloth)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG['model_name'],
    max_seq_length=CONFIG['max_seq_length'],
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG['lora_r'],
    lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=CONFIG['lora_dropout'],
    bias='none',
    target_modules=CONFIG['target_modules'],
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
)

model.print_trainable_parameters()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.15: Fast Qwen2 patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

---
## 5. Training

In [9]:
from transformers import TrainingArguments, Trainer as _HFTrainer
from trl import SFTTrainer

# ============ Fix: Unsloth / transformers >=4.46 compatibility ============
# transformers passes num_items_in_batch as int; Unsloth calls .mean() on it.
# Strategy A — class-level patch on Trainer.training_step (survives compiled cache)
_orig_trainer_step = _HFTrainer.training_step

def _safe_training_step(self, model, inputs, num_items_in_batch=None):
    if isinstance(num_items_in_batch, (int, float)):
        num_items_in_batch = torch.tensor(float(num_items_in_batch))
    return _orig_trainer_step(self, model, inputs, num_items_in_batch)

_HFTrainer.training_step = _safe_training_step
print("[patch] Trainer.training_step wrapped for int→tensor fix")
# ==========================================================================

training_args = TrainingArguments(
    output_dir=CONFIG['output_dir'],
    num_train_epochs=CONFIG['num_train_epochs'],
    per_device_train_batch_size=CONFIG['per_device_train_batch_size'],
    gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
    learning_rate=CONFIG['learning_rate'],
    warmup_ratio=CONFIG['warmup_ratio'],
    weight_decay=CONFIG['weight_decay'],
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=CONFIG['logging_steps'],
    eval_strategy='steps',
    eval_steps=CONFIG['eval_steps'],
    save_steps=CONFIG['save_steps'],
    save_total_limit=2,
    report_to='none',
    optim='paged_adamw_8bit',
    lr_scheduler_type='cosine',
    seed=SEED,
)

_resp_tpl = '<|im_start|>assistant\n'
print(f'[info] response_template = {_resp_tpl!r}')

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_text,
    eval_dataset=eval_text,
    dataset_text_field='text',
    max_seq_length=CONFIG['max_seq_length'],
    packing=False,
    args=training_args,
    response_template=_resp_tpl,
)

# Strategy B — disable num_items_in_batch from being passed at all
trainer.model_accepts_loss_kwargs = False
print("[patch] model_accepts_loss_kwargs = False  →  num_items_in_batch disabled")

train_result = trainer.train()
print(train_result)

TrainOutput(global_step=5, training_loss=0.03014965057373047, metrics={'train_runtime': 186.3107, 'train_samples_per_second': 1.675, 'train_steps_per_second': 0.027, 'total_flos': 2979070373167104.0, 'train_loss': 0.03014965057373047, 'epoch': 1.0})


In [10]:
os.makedirs(CONFIG['output_dir'], exist_ok=True)
trainer.save_model(CONFIG['output_dir'])
tokenizer.save_pretrained(CONFIG['output_dir'])
print(f'Adapter + tokenizer saved to {CONFIG["output_dir"]}')

Adapter + tokenizer saved to /kaggle/working/qwen25_3b_svg_lora


In [11]:
import gc

# Free training artifacts — optimizer states, gradients, cached activations
del trainer, training_args
gc.collect()
torch.cuda.empty_cache()

allocated = torch.cuda.memory_allocated() / 1e9
reserved = torch.cuda.memory_reserved() / 1e9
print(f"GPU memory after cleanup: {allocated:.2f} GB allocated, {reserved:.2f} GB reserved")

GPU memory after cleanup: 2.58 GB allocated, 2.65 GB reserved


---
## Phase B: Inference & Submission

> Everything below this line can run **offline** on Kaggle.
> For the actual Kaggle code submission, create a separate notebook that loads the pre-trained adapter from a Kaggle dataset and only executes the cells below.

In [ ]:
import gc, importlib
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# ── Step 1: Undo Unsloth's global monkey-patches ──
import transformers.models.qwen2.modeling_qwen2 as _qwen2
importlib.reload(_qwen2)
print('[fix] Reloaded qwen2 module')

try:
    del model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

# ── Step 2: Load base model fp16 ──
base_model = AutoModelForCausalLM.from_pretrained(
    CONFIG['model_name'],
    device_map="auto",
    torch_dtype=torch.float16,
)

# ── Step 3: Swap ALL Unsloth-patched classes → clean HF classes ──
base_model.__class__ = _qwen2.Qwen2ForCausalLM
base_model.model.__class__ = _qwen2.Qwen2Model
for _layer in base_model.model.layers:
    _layer.__class__ = _qwen2.Qwen2DecoderLayer
    _layer.self_attn.__class__ = _qwen2.Qwen2Attention
    _layer.mlp.__class__ = _qwen2.Qwen2MLP
    _layer.input_layernorm.__class__ = _qwen2.Qwen2RMSNorm
    _layer.post_attention_layernorm.__class__ = _qwen2.Qwen2RMSNorm
base_model.model.norm.__class__ = _qwen2.Qwen2RMSNorm
base_model.config._attn_implementation = "sdpa"
print(f'[fix] Swapped {len(base_model.model.layers)} layers → clean HF + SDPA')

# ── Step 4: Load LoRA, then MERGE into base weights ──
# PeftModel wrapping adds heavy per-token Python overhead (252 LoRA layers).
# merge_and_unload() fuses LoRA into base weights → plain model, zero overhead.
peft_model = PeftModel.from_pretrained(base_model, CONFIG['output_dir'])
model = peft_model.merge_and_unload()
print(f'[fix] LoRA merged into base weights — PeftModel overhead eliminated')

model.eval()
model.generation_config.max_length = None

# ── Step 5: Clear any leftover hooks from Unsloth ──
for _m in model.modules():
    _m._forward_hooks.clear()
    _m._forward_pre_hooks.clear()
print(f'[fix] Cleared all forward hooks')

# ── Verify ──
_impl = getattr(model.config, '_attn_implementation', 'unknown')
_dtype = next(model.parameters()).dtype
_on_gpu = all(p.device.type == 'cuda' for p in model.parameters())
print(f'\nModel class: {type(model).__name__}')
print(f'attn_implementation: {_impl}')
print(f'dtype: {_dtype}')
print(f'All params on GPU: {_on_gpu}')
assert _impl == "sdpa", f'Expected sdpa, got {_impl}'
assert _dtype == torch.float16, f'Expected fp16, got {_dtype}'
assert _on_gpu, 'Some params on CPU! Check device_map.'

tokenizer = AutoTokenizer.from_pretrained(CONFIG['output_dir'])
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ── EOS tokens ──
_eos_ids = []
if hasattr(model.config, 'eos_token_id'):
    v = model.config.eos_token_id
    _eos_ids = v if isinstance(v, list) else [v]
_im_end_id = tokenizer.convert_tokens_to_ids('<|im_end|>')
if _im_end_id and _im_end_id not in _eos_ids:
    _eos_ids.append(_im_end_id)
EOS_TOKEN_IDS = _eos_ids

alloc = torch.cuda.memory_allocated() / 1e9
print(f'EOS token IDs: {EOS_TOKEN_IDS}')
print(f'GPU mem: {alloc:.2f} GB')

# ── Quick speed benchmark (single forward pass) ──
_test_ids = tokenizer("hello", return_tensors="pt").input_ids.to(model.device)
torch.cuda.synchronize()
_t0 = time.time()
with torch.no_grad():
    for _ in range(10):
        model(_test_ids)
torch.cuda.synchronize()
_ms = (time.time() - _t0) / 10 * 1000
print(f'Forward pass latency: {_ms:.1f} ms (should be < 10 ms on A100)')

# ── Inference helpers ──

SVG_EXTRACT_RE = re.compile(r'<svg.*?</svg>', flags=re.IGNORECASE | re.DOTALL)

GENERATE_KWARGS = dict(
    max_new_tokens=CONFIG['max_new_tokens'],
    do_sample=True,
    temperature=CONFIG['temperature'],
    top_p=CONFIG['top_p'],
    repetition_penalty=CONFIG['repetition_penalty'],
    eos_token_id=EOS_TOKEN_IDS,
)


def extract_svg(text):
    m = SVG_EXTRACT_RE.search(text)
    return m.group(0).strip() if m else ''


def _postprocess(decoded, prompt):
    raw = extract_svg(decoded)
    if not raw:
        return fallback_svg(prompt), 'extract_fail'
    cleaned = sanitize_svg(raw)
    if not cleaned:
        return fallback_svg(prompt), 'sanitize_fail'
    ok, reason = validate_svg(cleaned)
    if not ok:
        return fallback_svg(prompt), f'validate_fail:{reason}'
    return cleaned, 'ok'


def generate_svg(prompt):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': prompt},
    ]
    chat = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(chat, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, **GENERATE_KWARGS)
    gen_ids = out[0][inputs['input_ids'].shape[-1]:]
    decoded = tokenizer.decode(gen_ids, skip_special_tokens=True)
    print(f'  [debug] generated {len(gen_ids)} tokens, decoded len={len(decoded)}')
    print(f'  [debug] first 300 chars: {decoded[:300]}')
    return _postprocess(decoded, prompt)


def generate_svg_batch(prompts):
    bs = CONFIG.get('inference_batch_size', 8)
    all_results = []

    for i in range(0, len(prompts), bs):
        batch = prompts[i:i + bs]
        chats = []
        for p in batch:
            msgs = [
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': p},
            ]
            chats.append(tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=True,
            ))

        inputs = tokenizer(
            chats, return_tensors='pt', padding=True,
            truncation=True, max_length=CONFIG['max_seq_length'],
        ).to(model.device)

        with torch.no_grad():
            out = model.generate(**inputs, **GENERATE_KWARGS)

        input_len = inputs['input_ids'].shape[-1]
        for j, prompt in enumerate(batch):
            gen_ids = out[j][input_len:]
            decoded = tokenizer.decode(gen_ids, skip_special_tokens=True)
            all_results.append(_postprocess(decoded, prompt))

    return all_results


In [13]:
# ═══════════════════════════════════════════════════════════
# SMOKE TEST — verify everything works BEFORE spending GPU
# hours on submission. If any check fails, FIX IT FIRST.
# ═══════════════════════════════════════════════════════════

print('=' * 60)
print('SMOKE TEST: Pre-submission Diagnostics')
print('=' * 60)

# ── 1. Environment check ──
_impl = getattr(model.config, '_attn_implementation', 'unknown')
_cls = type(model.base_model.model).__name__
_dtype = next(model.parameters()).dtype
_dev = next(model.parameters()).device
print(f'\n[ENV] Model class: {_cls}')
print(f'[ENV] attn_implementation: {_impl}')
print(f'[ENV] Model dtype: {_dtype}')
print(f'[ENV] Device: {_dev}')
print(f'[ENV] EOS token IDs: {EOS_TOKEN_IDS}')
print(f'[ENV] Batch size: {CONFIG.get("inference_batch_size", 8)}')
print(f'[ENV] max_new_tokens: {CONFIG["max_new_tokens"]}')

if _impl != "sdpa":
    print(f'\n*** WARNING: attn_implementation = "{_impl}", expected "sdpa" ***')
    print('    Inference will be slow. Fix Cell 17.')
else:
    print('\n[OK] SDPA is active.')

# ── 2. Single-prompt speed test ──
test_prompts = [
    'a simple red circle on white background',
    'a green tree with brown trunk',
    'a blue five-pointed star icon',
]

print(f'\n{"─" * 60}')
print(f'Running {len(test_prompts)} test prompts (single mode)...\n')

results = []
t0 = time.time()

for p in test_prompts:
    t1 = time.time()
    svg, status = generate_svg(p)
    elapsed = time.time() - t1
    results.append((p, svg, status, elapsed))
    print(f'  [{status:>20}] {elapsed:5.1f}s  len={len(svg):>5}  prompt="{p}"')

single_elapsed = time.time() - t0
print(f'\n  Total: {single_elapsed:.1f}s for {len(test_prompts)} prompts')

# ── 3. Batch speed test ──
bs = CONFIG.get('inference_batch_size', 8)
print(f'\n{"─" * 60}')
print(f'Running batch test (batch_size={bs})...\n')

t0 = time.time()
batch_results = generate_svg_batch(test_prompts)
batch_elapsed = time.time() - t0

for (p, (svg, status)) in zip(test_prompts, batch_results):
    print(f'  [{status:>20}] len={len(svg):>5}  prompt="{p}"')

print(f'\n  Batch total: {batch_elapsed:.1f}s for {len(test_prompts)} prompts')

# ── 4. Estimate submission time ──
per_prompt_sec = batch_elapsed / len(test_prompts)
total_prompts = 1000
est_minutes = (per_prompt_sec * total_prompts) / 60

print(f'\n{"═" * 60}')
print(f'SUBMISSION ESTIMATE')
print(f'  Speed:    {per_prompt_sec:.2f} s/prompt')
print(f'  Prompts:  {total_prompts}')
print(f'  ETA:      {est_minutes:.0f} min ({est_minutes/60:.1f} hours)')
print(f'{"═" * 60}')

# ── 5. Quality summary ──
valid = sum(1 for _, s in batch_results if s == 'ok')
print(f'\n[QUALITY] Valid: {valid}/{len(batch_results)}')
for p, (svg, status) in zip(test_prompts, batch_results):
    if status == 'ok':
        print(f'  OK  "{p}" -> {len(svg)} chars')
    else:
        print(f'  FAIL "{p}" -> {status}')

if est_minutes > 120:
    print(f'\n*** WARNING: Estimated {est_minutes:.0f} min is too slow! ***')
    print('    Expected: < 30 min on A100. Check Cell 17.')
elif est_minutes > 60:
    print(f'\nEstimated {est_minutes:.0f} min — acceptable but could be faster.')
else:
    print(f'\nSpeed looks good for submission.')


---
## 7. Generate Submission

In [14]:
test_df = pd.read_csv(CONFIG['test_prompts_path'])
print(f'Test prompts: {len(test_df)}')

# Suppress the noisy "max_new_tokens vs max_length" warning printed every generate() call
model.generation_config.max_length = None

all_prompts = test_df['prompt'].tolist()
all_ids = test_df['id'].tolist()
BS = CONFIG.get('inference_batch_size', 8)

rows = []
stats = Counter()
t0 = time.time()

for i in range(0, len(all_prompts), BS):
    batch_prompts = all_prompts[i:i + BS]
    batch_ids = all_ids[i:i + BS]
    batch_results = generate_svg_batch(batch_prompts)

    for pid, (svg, status) in zip(batch_ids, batch_results):
        stats[f'status:{status}'] += 1
        if status != 'ok':
            stats['fallback'] += 1
        else:
            stats['valid'] += 1
        stats['total_len'] += len(svg)
        rows.append({'id': pid, 'svg': svg})

    done = min(i + BS, len(all_prompts))
    if done % 50 < BS or done >= len(all_prompts):
        elapsed = time.time() - t0
        rate = done / max(elapsed, 1)
        eta = (len(all_prompts) - done) / max(rate, 0.01)
        print(f'  [{done}/{len(all_prompts)}] {elapsed:.0f}s '
              f'({rate:.2f} it/s, ETA {eta/60:.0f}min)  '
              f'valid={stats["valid"]}  fallback={stats["fallback"]}')

sub_df = pd.DataFrame(rows)
sub_df.to_csv(CONFIG['submission_path'], index=False)

elapsed_min = (time.time() - t0) / 60
n = len(sub_df)
print(f'\nSubmission saved: {CONFIG["submission_path"]}')
print(f'Total rows:    {n}')
print(f'Valid:         {stats["valid"]} ({100*stats["valid"]/max(n,1):.1f}%)')
print(f'Fallback:      {stats["fallback"]} ({100*stats["fallback"]/max(n,1):.1f}%)')
print(f'Avg SVG len:   {stats["total_len"]/max(n,1):.0f} chars')
print(f'Runtime:       {elapsed_min:.1f} min')

print(f'\n--- Failure distribution ---')
for k, v in sorted(stats.items()):
    if k.startswith('status:'):
        print(f'  {k}: {v}')

sub_df.head()

Both `max_new_tokens` (=768) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Test prompts: 1000


Both `max_new_tokens` (=768) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=768) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=768) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=768) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

  [52/1000] 1168s (0.04 it/s, ETA 355min)  valid=0  fallback=52


Both `max_new_tokens` (=768) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=768) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=768) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=768) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

...

Both `max_new_tokens` (=768) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=768) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=768) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=768) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

  [952/1000] 23489s (0.04 it/s, ETA 20min)  valid=0  fallback=952


Both `max_new_tokens` (=768) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=768) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=768) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=768) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

  [1000/1000] 24534s (0.04 it/s, ETA 0min)  valid=0  fallback=1000

Submission saved: /kaggle/working/submission.csv
Total rows:    1000
Valid:         0 (0.0%)
Fallback:      1000 (100.0%)
Avg SVG len:   196 chars
Runtime:       408.9 min

--- Failure distribution ---
  status:extract_fail: 262
  status:validate_fail:parse_error: duplicate attribute: line 1, column 87: 738


,id,svg
0,fa1d8fa7-080f-4269-a9cf-a17562c9a0ca,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
1,6eede943219547c22ac56085027d33cc,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
2,ea045c7a247166f061ce504d9b7ccaab,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
3,8fe82f3af89e487b31236ca829c3f071,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
4,600464e4d92c75338462271a09b3f176,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."


---
## Summary & Notes

**What this baseline does:**

- Loads competition `train.csv` (no external datasets), normalizes SVGs to 256×256, filters by quality (≤2 500 chars)
- Fine-tunes Qwen2.5-3B-Instruct with QLoRA (`r=16`, 4-bit, `packing=False`, `response_template` for loss masking)
- Batched inference (`inference_batch_size=4`) with conservative decoding (`temperature=0.3`, `top_p=0.8`)
- Full post-processing chain: extract → sanitize → validate → fallback
- Tracks per-layer failure reasons (`extract_fail`, `sanitize_fail`, `validate_fail:<reason>`)

**Two-phase design:**

- **Phase A (Training):** Cells 1-15. Requires internet to load model weights. Saves LoRA adapter to `output_dir`.
- **Phase B (Inference):** Cells 16-21. Reloads model without Unsloth patches for clean generation.

**Key parameters to tune next:**

| Parameter | Current | Try |
|---|---|---|
| `max_train_samples` | 12 000 | 20 000-29 000 |
| `max_svg_train_chars` | 2 500 | 3 000-4 000 (with monitoring) |
| `lora_r` | 16 | 32, 64 |
| `num_train_epochs` | 1 | 2-3 |
| `temperature` | 0.3 | 0.5-0.7 (after valid rate ≥ 80%) |
| `inference_batch_size` | 4 | 8 (if VRAM allows) |